# CarSales Miner Summary

This notebook browses CarSales miner outputs with filters tailored to seller-response deception.

It is tuned for:
- `DISCLOSE_ISSUE` vs `CONCEAL_ISSUE`
- current round focus and target issue
- omission / false-denial label details
- seller message, reasoning, prompt, and secondary events


In [ ]:
from pathlib import Path
import html
import json

import ipywidgets as widgets
import pandas as pd
from IPython.display import HTML, JSON, Markdown, clear_output, display

pd.set_option('display.max_colwidth', 200)

repo_root = Path('/playpen-ssd/smerrill/deception2')
data_root = repo_root / 'CarSales' / 'Results' / 'deception_miner' / 'deepseek-ai_DeepSeek-R1-Distill-Qwen-7B'
data_root

In [ ]:
jsonl_files = sorted(data_root.rglob('deception_samples.jsonl'))
print(f'Found {len(jsonl_files)} jsonl files under {data_root}')


def exact_action_type(value):
    if value is None:
        return None
    text = str(value).strip()
    return text if text else None


def deceptive_from_action_type(action_type):
    if action_type == 'CONCEAL_ISSUE':
        return True
    if action_type == 'DISCLOSE_ISSUE':
        return False
    return None


def deception_reason_from_action_type(action_type):
    if action_type == 'CONCEAL_ISSUE':
        return 'seller_chose_conceal_issue'
    if action_type == 'DISCLOSE_ISSUE':
        return 'seller_chose_disclose_issue'
    if action_type is None:
        return 'seller_action_missing'
    return 'seller_action_unrecognized'


records = []
for path in jsonl_files:
    run_tag = path.parent.parent.name
    shard = path.parent.name
    meta_path = path.parent / 'meta.json'
    meta = {}
    if meta_path.exists():
        try:
            meta = json.loads(meta_path.read_text(encoding='utf-8'))
        except Exception:
            meta = {}

    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except Exception:
                continue

            action = rec.get('action', {}) if isinstance(rec.get('action'), dict) else {}
            truth_context = rec.get('truth_context', {}) if isinstance(rec.get('truth_context'), dict) else {}
            label_preview = truth_context.get('label_preview', {}) if isinstance(truth_context.get('label_preview'), dict) else {}
            messages_obj = rec.get('messages', []) if isinstance(rec.get('messages'), list) else []
            secondary_events = rec.get('secondary_events', []) if isinstance(rec.get('secondary_events'), list) else []

            raw_action_type = exact_action_type(action.get('Action'))
            seller_action_kind = exact_action_type(truth_context.get('seller_action_kind') or label_preview.get('seller_action_kind'))
            effective_action_type = raw_action_type if raw_action_type is not None else seller_action_kind
            message = action.get('Message')
            reasoning = action.get('reasoning') or action.get('_raw_text')
            target_issue_id = rec.get('target_issue_id') or truth_context.get('target_issue_id') or label_preview.get('target_issue_id')
            current_round_focus = rec.get('current_round_focus') or truth_context.get('current_round_focus') or label_preview.get('question_focus')
            recorded_deceptive = rec.get('deceptive')
            deceptive = deceptive_from_action_type(raw_action_type)
            if deceptive is None and raw_action_type is None:
                deceptive = deceptive_from_action_type(seller_action_kind)
            deceptive_label = 'DECEPTIVE' if deceptive is True else ('TRUTHFUL' if deceptive is False else 'UNKNOWN')
            deception_reason = deception_reason_from_action_type(effective_action_type)

            records.append({
                'source_path': str(path),
                'run_tag': run_tag,
                'shard': shard,
                'model_name': meta.get('model_name'),
                'scenario_name': rec.get('scenario_name'),
                'state_id': rec.get('state_id'),
                'game_id': rec.get('game_id'),
                'turn_idx': rec.get('turn_idx'),
                'sample_idx': rec.get('sample_idx'),
                'seed': rec.get('seed'),
                'resample_round_idx': rec.get('resample_round_idx'),
                'deceptive': deceptive,
                'recorded_deceptive': recorded_deceptive,
                'deceptive_label': deceptive_label,
                'used_for_transition': rec.get('used_for_transition'),
                'phase': rec.get('phase'),
                'raw_action_type': raw_action_type,
                'action_type': effective_action_type,
                'message': message,
                'reasoning': reasoning,
                'current_round_focus': current_round_focus,
                'target_issue_id': target_issue_id,
                'seller_action_kind': seller_action_kind,
                'deception_reason': deception_reason,
                'false_denial_detected': label_preview.get('false_denial_detected'),
                'omitted_issue_ids': label_preview.get('omitted_issue_ids'),
                'newly_disclosed_issue_ids': label_preview.get('newly_disclosed_issue_ids'),
                'delayed_disclosure_issue_ids': label_preview.get('delayed_disclosure_issue_ids'),
                'current_disclosed_issue_ids': label_preview.get('current_disclosed_issue_ids'),
                'disclosed_issue_ids_by_now': label_preview.get('disclosed_issue_ids_by_now'),
                'relevant_known_issue_ids': label_preview.get('relevant_known_issue_ids'),
                'already_disclosed_issue_ids': truth_context.get('already_disclosed_issue_ids'),
                'truthful_action': truth_context.get('truthful_action'),
                'full_action': action,
                'truth_context': truth_context,
                'label_preview': label_preview,
                'messages_obj': messages_obj,
                'secondary_events': secondary_events,
                'prompt': rec.get('prompt'),
            })

df = pd.DataFrame(records)
print(f'Loaded {len(df)} records')
df.head(3)


In [ ]:

total = len(df)
deceptive_total = int((df['deceptive'] == True).sum()) if total else 0
truthful_total = int((df['deceptive'] == False).sum()) if total else 0
unknown_total = int(df['deceptive'].isna().sum()) if total else 0

summary_df = pd.DataFrame([
    {'metric': 'total_records', 'value': total},
    {'metric': 'deceptive_records', 'value': deceptive_total},
    {'metric': 'truthful_records', 'value': truthful_total},
    {'metric': 'unknown_records', 'value': unknown_total},
    {'metric': 'jsonl_files', 'value': len(jsonl_files)},
    {'metric': 'scenarios', 'value': 0 if df.empty else df['scenario_name'].nunique(dropna=True)},
])
display(summary_df)

if df.empty:
    display(Markdown(f'No CarSales miner records found under `{data_root}` yet.'))
else:
    display(Markdown('### Action Type Counts'))
    display(df['action_type'].fillna('(missing)').value_counts().rename_axis('action_type').reset_index(name='count'))

    display(Markdown('### Round Focus Counts'))
    display(df['current_round_focus'].fillna('(missing)').value_counts().rename_axis('current_round_focus').reset_index(name='count'))

    preview_cols = ['scenario_name', 'deceptive_label', 'current_round_focus', 'target_issue_id', 'action_type', 'message']
    display(Markdown('### Preview'))
    display(df[preview_cols].head(10))


In [ ]:

if df.empty:
    print(f'No records to browse under {data_root}')
else:
    browser_df = df.copy().reset_index(drop=True)
    browser_df['scenario_name'] = browser_df['scenario_name'].fillna('(missing)')
    browser_df['action_type'] = browser_df['action_type'].fillna('(missing)')
    browser_df['raw_action_type'] = browser_df['raw_action_type'].fillna('(missing)')
    browser_df['current_round_focus'] = browser_df['current_round_focus'].fillna('(missing)')
    browser_df['target_issue_id'] = browser_df['target_issue_id'].fillna('(missing)')
    browser_df['message'] = browser_df['message'].fillna('')
    browser_df['reasoning'] = browser_df['reasoning'].fillna('')
    browser_df['deception_reason'] = browser_df['deception_reason'].fillna('')
    browser_df['message_preview'] = browser_df['message'].str.replace('\n', ' ', regex=False).str.slice(0, 100)

    def _list_text(value):
        if isinstance(value, list):
            return ', '.join(str(x) for x in value)
        return '' if value is None else str(value)

    browser_df['omitted_issue_ids_text'] = browser_df['omitted_issue_ids'].apply(_list_text)
    browser_df['newly_disclosed_issue_ids_text'] = browser_df['newly_disclosed_issue_ids'].apply(_list_text)
    browser_df['current_disclosed_issue_ids_text'] = browser_df['current_disclosed_issue_ids'].apply(_list_text)
    browser_df['disclosed_issue_ids_by_now_text'] = browser_df['disclosed_issue_ids_by_now'].apply(_list_text)
    browser_df['relevant_known_issue_ids_text'] = browser_df['relevant_known_issue_ids'].apply(_list_text)
    browser_df['dropdown_label'] = browser_df.apply(
        lambda row: (
            f"{row.name:04d} | {row['deceptive_label']} | {row['scenario_name']} | "
            f"{row['current_round_focus']} | {row['action_type']} | {row['target_issue_id']} | {row['message_preview']}"
        ),
        axis=1,
    )

    label_filter = widgets.Dropdown(
        options=['ALL', 'DECEPTIVE', 'TRUTHFUL', 'UNKNOWN'],
        value='ALL',
        description='Label:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='220px'),
    )
    scenario_filter = widgets.Dropdown(
        options=['ALL'] + sorted(str(x) for x in browser_df['scenario_name'].dropna().unique()),
        value='ALL',
        description='Scenario:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='280px'),
    )
    action_filter = widgets.Dropdown(
        options=['ALL'] + sorted(str(x) for x in browser_df['action_type'].dropna().unique()),
        value='ALL',
        description='Action:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='240px'),
    )
    focus_filter = widgets.Dropdown(
        options=['ALL'] + sorted(str(x) for x in browser_df['current_round_focus'].dropna().unique()),
        value='ALL',
        description='Focus:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='260px'),
    )
    issue_filter = widgets.Dropdown(
        options=['ALL'] + sorted(str(x) for x in browser_df['target_issue_id'].dropna().unique()),
        value='ALL',
        description='Target issue:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='280px'),
    )
    transition_filter = widgets.Dropdown(
        options=['ALL', 'USED_FOR_TRANSITION', 'NOT_USED_FOR_TRANSITION'],
        value='ALL',
        description='Transition:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='280px'),
    )
    search_box = widgets.Text(
        value='',
        description='Search:',
        placeholder='message / reasoning / issue ids / raw action',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='480px'),
    )
    example_dropdown = widgets.Dropdown(
        options=[],
        description='Example:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='100%'),
    )
    show_prompt = widgets.Checkbox(value=False, description='Show prompt')
    show_messages = widgets.Checkbox(value=False, description='Show messages')
    show_truth_context = widgets.Checkbox(value=True, description='Show truth context')
    show_secondary_events = widgets.Checkbox(value=False, description='Show secondary events')
    out = widgets.Output()
    state = {'filtered': browser_df}

    def _filtered_df():
        filtered = browser_df.copy()

        if label_filter.value == 'DECEPTIVE':
            filtered = filtered[filtered['deceptive'] == True]
        elif label_filter.value == 'TRUTHFUL':
            filtered = filtered[filtered['deceptive'] == False]
        elif label_filter.value == 'UNKNOWN':
            filtered = filtered[filtered['deceptive'].isna()]

        if scenario_filter.value != 'ALL':
            filtered = filtered[filtered['scenario_name'].astype(str) == scenario_filter.value]

        if action_filter.value != 'ALL':
            filtered = filtered[filtered['action_type'].astype(str) == action_filter.value]

        if focus_filter.value != 'ALL':
            filtered = filtered[filtered['current_round_focus'].astype(str) == focus_filter.value]

        if issue_filter.value != 'ALL':
            filtered = filtered[filtered['target_issue_id'].astype(str) == issue_filter.value]

        if transition_filter.value == 'USED_FOR_TRANSITION':
            filtered = filtered[filtered['used_for_transition'] == True]
        elif transition_filter.value == 'NOT_USED_FOR_TRANSITION':
            filtered = filtered[filtered['used_for_transition'] != True]

        needle = search_box.value.strip().lower()
        if needle:
            mask = (
                filtered['scenario_name'].fillna('').str.lower().str.contains(needle, regex=False)
                | filtered['message'].fillna('').str.lower().str.contains(needle, regex=False)
                | filtered['reasoning'].fillna('').str.lower().str.contains(needle, regex=False)
                | filtered['target_issue_id'].fillna('').str.lower().str.contains(needle, regex=False)
                | filtered['current_round_focus'].fillna('').str.lower().str.contains(needle, regex=False)
                | filtered['action_type'].fillna('').str.lower().str.contains(needle, regex=False)
                | filtered['raw_action_type'].fillna('').str.lower().str.contains(needle, regex=False)
                | filtered['deception_reason'].fillna('').str.lower().str.contains(needle, regex=False)
                | filtered['omitted_issue_ids_text'].fillna('').str.lower().str.contains(needle, regex=False)
                | filtered['newly_disclosed_issue_ids_text'].fillna('').str.lower().str.contains(needle, regex=False)
                | filtered['current_disclosed_issue_ids_text'].fillna('').str.lower().str.contains(needle, regex=False)
                | filtered['relevant_known_issue_ids_text'].fillna('').str.lower().str.contains(needle, regex=False)
            )
            filtered = filtered[mask]

        return filtered.reset_index(drop=True)

    def _render_json_block(value):
        if value in (None, '', [], {}):
            display(Markdown('_None_'))
        else:
            display(JSON(value))

    def _render_pre_block(title, text):
        display(Markdown(f'### {title}'))
        if not text:
            display(Markdown('_None_'))
            return
        display(HTML(f"<pre style='white-space: pre-wrap; font-family: monospace;'>{html.escape(str(text))}</pre>"))

    def _render(*_):
        with out:
            clear_output()
            filtered = state['filtered']
            if filtered.empty or example_dropdown.value is None:
                display(Markdown('No CarSales examples match the current filters.'))
                return

            row = filtered.iloc[int(example_dropdown.value)]
            meta_rows = pd.DataFrame([
                {'field': 'scenario_name', 'value': row['scenario_name']},
                {'field': 'run_tag', 'value': row['run_tag']},
                {'field': 'shard', 'value': row['shard']},
                {'field': 'state_id', 'value': row['state_id']},
                {'field': 'game_id', 'value': row['game_id']},
                {'field': 'turn_idx', 'value': row['turn_idx']},
                {'field': 'sample_idx', 'value': row['sample_idx']},
                {'field': 'seed', 'value': row['seed']},
                {'field': 'resample_round_idx', 'value': row['resample_round_idx']},
                {'field': 'deceptive', 'value': row['deceptive']},
                {'field': 'recorded_deceptive', 'value': row['recorded_deceptive']},
                {'field': 'action_type', 'value': row['action_type']},
                {'field': 'raw_action_type', 'value': row['raw_action_type']},
                {'field': 'used_for_transition', 'value': row['used_for_transition']},
                {'field': 'current_round_focus', 'value': row['current_round_focus']},
                {'field': 'target_issue_id', 'value': row['target_issue_id']},
                {'field': 'deception_reason', 'value': row['deception_reason']},
                {'field': 'false_denial_detected', 'value': row['false_denial_detected']},
                {'field': 'omitted_issue_ids', 'value': row['omitted_issue_ids_text']},
                {'field': 'newly_disclosed_issue_ids', 'value': row['newly_disclosed_issue_ids_text']},
                {'field': 'current_disclosed_issue_ids', 'value': row['current_disclosed_issue_ids_text']},
                {'field': 'disclosed_issue_ids_by_now', 'value': row['disclosed_issue_ids_by_now_text']},
            ])
            display(meta_rows)

            _render_pre_block('Seller Message', row['message'])
            _render_pre_block('Reasoning', row['reasoning'])

            display(Markdown('### Full Action'))
            _render_json_block(row['full_action'])

            display(Markdown('### Label Preview'))
            _render_json_block(row['label_preview'])

            display(Markdown('### Truthful Action Reference'))
            _render_json_block(row['truthful_action'])

            if show_truth_context.value:
                display(Markdown('### Truth Context'))
                _render_json_block(row['truth_context'])

            if show_prompt.value:
                _render_pre_block('Prompt', row['prompt'])

            if show_messages.value:
                display(Markdown('### Messages'))
                _render_json_block(row['messages_obj'])

            if show_secondary_events.value:
                display(Markdown('### Secondary Events'))
                _render_json_block(row['secondary_events'])

    def _update_options(*_):
        filtered = _filtered_df()
        state['filtered'] = filtered
        if filtered.empty:
            example_dropdown.options = []
            example_dropdown.value = None
            _render()
            return

        options = [(label, idx) for idx, label in enumerate(filtered['dropdown_label'])]
        example_dropdown.options = options
        if example_dropdown.value is None or example_dropdown.value >= len(filtered):
            example_dropdown.value = 0
        _render()

    for widget in [
        label_filter,
        scenario_filter,
        action_filter,
        focus_filter,
        issue_filter,
        transition_filter,
        search_box,
        example_dropdown,
        show_prompt,
        show_messages,
        show_truth_context,
        show_secondary_events,
    ]:
        widget.observe(_update_options, names='value')

    controls_top = widgets.HBox([label_filter, scenario_filter, action_filter])
    controls_mid = widgets.HBox([focus_filter, issue_filter, transition_filter])
    toggles = widgets.HBox([show_prompt, show_messages, show_truth_context, show_secondary_events])
    display(controls_top)
    display(controls_mid)
    display(search_box)
    display(example_dropdown)
    display(toggles)
    display(out)
    _update_options()
